# 18.6 绿色 AI 与能效优化 (Green AI & Energy Efficiency)

> 🕐 预估学习时间：30分钟

本节聚焦大语言模型训练与推理过程中的能耗与碳排放问题。随着模型规模指数级增长，单次训练动辄消耗兆瓦时级电力，推理服务在规模化部署后也成为重要碳排放源。绿色 AI 旨在通过算法、硬件与调度协同优化，在保证模型质量的前提下显著降低能耗与碳足迹。

**学习主题：**

1. AI 能耗概述与碳足迹核算（EnergyEstimator、CarbonFootprintCalculator 类）
2. 训练能效优化（TrainingEfficiencyAnalyzer 类）
3. 推理能效优化（InferenceEfficiencyAnalyzer 类）
4. 硬件选择与数据中心（HardwareComparator 类）
5. 可持续 AI 实践（SustainableAIPlanner 类）

## 1. AI 能耗概述

大模型训练成本常以百万美元计，背后是数千张 GPU 持续数月的高强度运算。能耗与碳排放已成为 AI 研究不可回避的议题。

**绿色 AI 的核心动机：**

- **训练成本高昂**：GPT-3 级别训练耗电约 1.28 GWh，相当于约 120 个家庭一年的用电量
- **推理规模放大**：规模化部署后，推理总能耗往往超过训练能耗
- **碳排放压力**：电力来源若以化石燃料为主，碳足迹显著上升
- **能源结构差异**：不同地区电网碳强度相差 10 倍以上

**能耗构成：**

- **GPU 计算能耗**：矩阵乘法与注意力计算占主要部分
- **内存与带宽**：HBM 显存读写、KV cache 访问
- **通信开销**：分布式训练中的 all-reduce、all-gather
- **数据中心基础设施**：冷却、供电、网络设备（PUE 通常 1.1-1.5）

**核算关键指标：**

- **能耗（kWh/MWh）**：GPU 数量 × 单卡功率 × 训练时长
- **碳强度（gCO2/kWh）**：每度电对应的 CO2 排放克数
- **PUE**：数据中心总能耗 / IT 设备能耗
- **性能功耗比**：每瓦特算力或每焦耳生成 token 数

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== AI 能耗与碳足迹估算 ===')

class EnergyEstimator:
    '''训练能耗估算器：基于 GPU 数量、时长与单卡功率估算总能耗'''

    def __init__(self, gpu_count, gpu_tdp_w, hours, pue=1.2):
        self.gpu_count = gpu_count
        self.gpu_tdp_w = gpu_tdp_w
        self.hours = hours
        self.pue = pue  # 数据中心能源效率

    def gpu_energy_kwh(self):
        '''仅 GPU 计算能耗'''
        return self.gpu_count * self.gpu_tdp_w * self.hours / 1000.0

    def total_energy_kwh(self):
        '''含 PUE 的总能耗（含冷却、供电损耗）'''
        return self.gpu_energy_kwh() * self.pue

    def total_energy_mwh(self):
        return self.total_energy_kwh() / 1000.0

    def describe(self):
        return OrderedDict([
            ('GPU 数量', self.gpu_count),
            ('单卡 TDP(W)', self.gpu_tdp_w),
            ('训练时长(h)', self.hours),
            ('PUE', self.pue),
            ('GPU 能耗(kWh)', round(self.gpu_energy_kwh(), 1)),
            ('总能耗(kWh)', round(self.total_energy_kwh(), 1)),
            ('总能耗(MWh)', round(self.total_energy_mwh(), 3)),
        ])

class CarbonFootprintCalculator:
    '''碳足迹计算器：将能耗转换为 CO2 排放'''

    # 各地区电网平均碳强度 gCO2/kWh
    REGION_INTENSITY = OrderedDict([
        ('挪威', 30),       # 几乎全水电
        ('法国', 60),       # 核电为主
        ('美国西部', 250),  # 风光+天然气
        ('美国平均', 380),
        ('中国华东', 580),  # 煤电占比高
        ('中国平均', 610),
        ('澳大利亚', 700),  # 煤电主导
    ])

    def __init__(self, energy_kwh, region='中国平均'):
        self.energy_kwh = energy_kwh
        self.region = region

    def co2_kg(self):
        intensity = self.REGION_INTENSITY[self.region]
        return self.energy_kwh * intensity / 1000.0

    def co2_tons(self):
        return self.co2_kg() / 1000.0

    def trees_equivalent(self):
        '''一棵树年均吸收约 21 kg CO2'''
        return self.co2_kg() / 21.0

    def describe(self):
        return OrderedDict([
            ('地区', self.region),
            ('碳强度(gCO2/kWh)', self.REGION_INTENSITY[self.region]),
            ('能耗(kWh)', round(self.energy_kwh, 1)),
            ('CO2(kg)', round(self.co2_kg(), 1)),
            ('CO2(吨)', round(self.co2_tons(), 3)),
            ('等效树木(棵)', round(self.trees_equivalent(), 0)),
        ])

# 估算不同规模模型训练能耗
scenarios = [
    ('1B 模型预训练', 8, 400, 168),      # 8 卡 1 周
    ('7B 模型预训练', 64, 400, 720),     # 64 卡 30 天
    ('70B 模型预训练', 512, 700, 1440),  # 512 卡 60 天
    ('175B 模型预训练', 1024, 700, 2160),# 1024 卡 90 天
]

print('\n--- 不同规模模型训练能耗 ---')
print(f"{'场景':<22}{'GPU数':>6}{'TDP(W)':>8}{'时长(h)':>9}{'总能耗(MWh)':>13}")
for name, gc, tdp, hrs in scenarios:
    est = EnergyEstimator(gc, tdp, hrs)
    print(f"{name:<22}{gc:>6}{tdp:>8}{hrs:>9}{est.total_energy_mwh():>13.2f}")

# 同一训练任务在不同地区的碳足迹对比
print('\n--- 175B 模型在不同地区的碳足迹对比 ---')
big_est = EnergyEstimator(1024, 700, 2160)
print(f"{'地区':<14}{'碳强度':>10}{'能耗(MWh)':>12}{'CO2(吨)':>10}{'等效树木':>10}")
for region in CarbonFootprintCalculator.REGION_INTENSITY:
    calc = CarbonFootprintCalculator(big_est.total_energy_kwh(), region)
    info = calc.describe()
    print(f"{region:<14}{info['碳强度(gCO2/kWh)']:>10}{info['能耗(kWh)']/1000:>12.1f}"
          f"{info['CO2(吨)']:>10.1f}{info['等效树木(棵)']:>10.0f}")

print(f'\nKey: 训练能耗随 GPU 数量与时长线性增长，但碳足迹高度依赖地区电网碳强度——同样的训练在挪威仅产生约 5% 于澳大利亚的碳排放，碳感知调度（选择低碳地区与时段）是绿色 AI 最直接的杠杆')

## 2. 训练能效优化

在不改变最终模型质量的前提下，通过训练策略优化可显著降低能耗。这些技术大多同时提升训练速度，是绿色 AI 的"免费午餐"。

**核心优化技术：**

- **混合精度训练**：FP16/BF16 相比 FP32 减半显存与算力开销，Tensor Core 加速 2-4 倍
- **梯度检查点**：以约 20% 额外计算换取显存大幅下降，允许更大 batch
- **高效数据加载**：预取、内存映射、tokenized 缓存避免 I/O 阻塞
- **最优 batch size**：过大导致显存溢出，过小浪费并行度
- **ZeRO/FSDP 分片**：减少冗余参数存储，提升显存利用率
- **通信优化**：梯度压缩、计算通信重叠

**精度权衡：**

- **FP32**：基线，最稳定但最慢最耗能
- **FP16**：速度快但需 loss scaling 防止下溢
- **BF16**：动态范围与 FP32 相同，训练最稳健
- **FP8**：最新标准，进一步降低显存与能耗，但对硬件要求高

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 训练能效分析 TrainingEfficiencyAnalyzer ===')

class TrainingEfficiencyAnalyzer:
    '''训练能效分析器：对比不同精度与优化策略的能耗成本'''

    # 精度配置：相对 FP32 的算力倍率、显存系数、稳定性、硬件支持
    PRECISION = OrderedDict([
        ('FP32', {'speedup': 1.0, 'mem_factor': 1.0, 'stability': 1.0, 'energy_factor': 1.0}),
        ('FP16', {'speedup': 2.0, 'mem_factor': 0.5, 'stability': 0.85, 'energy_factor': 0.55}),
        ('BF16', {'speedup': 2.0, 'mem_factor': 0.5, 'stability': 0.99, 'energy_factor': 0.55}),
        ('FP8',  {'speedup': 3.5, 'mem_factor': 0.25, 'stability': 0.92, 'energy_factor': 0.35}),
    ])

    # 优化策略：额外加速比、额外能耗系数
    STRATEGIES = OrderedDict([
        ('基线', {'speedup': 1.0, 'energy_factor': 1.0}),
        ('梯度检查点', {'speedup': 0.85, 'energy_factor': 1.20}),
        ('FlashAttention', {'speedup': 1.8, 'energy_factor': 0.65}),
        ('ZeRO-3', {'speedup': 1.3, 'energy_factor': 0.85}),
        ('组合优化', {'speedup': 2.5, 'energy_factor': 0.50}),
    ])

    def __init__(self, params_b, tokens_b, gpu_count=8, gpu_tdp_w=400, base_hours=240):
        self.params_b = params_b
        self.tokens_b = tokens_b
        self.gpu_count = gpu_count
        self.gpu_tdp_w = gpu_tdp_w
        self.base_hours = base_hours

    def estimate_flops(self):
        '''训练 FLOPs 估算：6 × 参数量 × token 数'''
        return 6 * self.params_b * 1e9 * self.tokens_b * 1e9

    def analyze_precision(self):
        '''对比不同精度的训练时间与能耗'''
        results = OrderedDict()
        for name, cfg in self.PRECISION.items():
            hours = self.base_hours / cfg['speedup']
            energy_kwh = self.gpu_count * self.gpu_tdp_w * hours / 1000.0
            energy_kwh *= cfg['energy_factor']
            results[name] = OrderedDict([
                ('precision', name),
                ('speedup', cfg['speedup']),
                ('hours', round(hours, 1)),
                ('energy_kwh', round(energy_kwh, 1)),
                ('stability', cfg['stability']),
            ])
        return results

    def analyze_strategies(self, precision='BF16'):
        '''在指定精度下对比优化策略'''
        pcfg = self.PRECISION[precision]
        results = OrderedDict()
        for name, cfg in self.STRATEGIES.items():
            hours = self.base_hours / (pcfg['speedup'] * cfg['speedup'])
            energy_kwh = self.gpu_count * self.gpu_tdp_w * hours / 1000.0
            energy_kwh *= pcfg['energy_factor'] * cfg['energy_factor']
            results[name] = OrderedDict([
                ('strategy', name),
                ('hours', round(hours, 1)),
                ('energy_kwh', round(energy_kwh, 1)),
                ('speedup_vs_baseline', round(pcfg['speedup'] * cfg['speedup'], 2)),
            ])
        return results

# 分析 7B 模型训练
analyzer = TrainingEfficiencyAnalyzer(params_b=7, tokens_b=2.0, gpu_count=64, gpu_tdp_w=400, base_hours=720)

print(f'\n--- 7B 模型训练 FLOPs 估算 ---')
flops = analyzer.estimate_flops()
print(f'  总 FLOPs: {flops:.2e}')
print(f'  约合 {flops/1e21:.2f} ZFLOPs')

print('\n--- 不同精度训练对比（7B, 64 GPU）---')
print(f"{'精度':<8}{'加速比':>8}{'时长(h)':>10}{'能耗(kWh)':>12}{'稳定性':>8}")
for name, info in analyzer.analyze_precision().items():
    print(f"{name:<8}{info['speedup']:>8.1f}x{info['hours']:>10.1f}{info['energy_kwh']:>12.1f}{info['stability']:>8.2f}")

print('\n--- BF16 精度下不同优化策略对比 ---')
print(f"{'策略':<14}{'总加速':>10}{'时长(h)':>10}{'能耗(kWh)':>12}")
for name, info in analyzer.analyze_strategies('BF16').items():
    print(f"{name:<14}{info['speedup_vs_baseline']:>9.2f}x{info['hours']:>10.1f}{info['energy_kwh']:>12.1f}")

# 节能潜力汇总
baseline_energy = analyzer.analyze_precision()['FP32']['energy_kwh']
best_energy = analyzer.analyze_strategies('BF16')['组合优化']['energy_kwh']
saving = (1 - best_energy / baseline_energy) * 100
print(f'\n--- 节能潜力 ---')
print(f'  FP32 基线能耗: {baseline_energy:.1f} kWh')
print(f'  BF16 + 组合优化: {best_energy:.1f} kWh')
print(f'  节能比例: {saving:.1f}%')

print(f'\nKey: BF16 是训练精度的甜点选择——速度与 FP16 相当但稳定性接近 FP32；叠加 FlashAttention、ZeRO 等组合优化可在 FP32 基线上节能 70% 以上，是绿色训练的标配组合')

## 3. 推理能效优化

推理能耗在模型生命周期总能耗中占比往往超过训练。规模化服务后，每节省 1% 的推理能耗都意味着可观的成本与碳减排。

**主要优化方向：**

- **模型量化**：INT8/INT4 推理减少显存占用与算力能耗
- **批处理**：合并请求提升 GPU 利用率，摊薄固定开销
- **动态推理**：根据输入难度调整计算量（early exit、MoE 路由）
- **KV cache 优化**：PagedAttention、prefix caching 减少重复计算
- **投机解码**：小模型生成草稿、大模型验证，减少大模型前向次数
- **模型蒸馏**：用小模型替代大模型处理简单请求

**关键指标：**

- **tokens/Joule**：每焦耳生成 token 数，衡量推理能效
- **tokens/s/GPU**：单卡吞吐
- **延迟-能耗积（EDP）**：延迟 × 能耗，综合衡量体验与成本

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 推理能效分析 InferenceEfficiencyAnalyzer ===')

class InferenceEfficiencyAnalyzer:
    '''推理能效分析器：对比不同优化策略的每焦耳 token 产出'''

    # 优化策略：相对基线的吞吐倍率、能耗系数、质量保留
    STRATEGIES = OrderedDict([
        ('FP16 基线', {'throughput': 1.0, 'energy_factor': 1.0, 'quality': 1.000}),
        ('INT8 量化', {'throughput': 1.8, 'energy_factor': 0.65, 'quality': 0.990}),
        ('INT4 量化', {'throughput': 2.8, 'energy_factor': 0.45, 'quality': 0.965}),
        ('动态批处理', {'throughput': 3.5, 'energy_factor': 1.10, 'quality': 1.000}),
        ('投机解码', {'throughput': 1.9, 'energy_factor': 0.85, 'quality': 1.000}),
        ('Early Exit', {'throughput': 1.5, 'energy_factor': 0.70, 'quality': 0.980}),
        ('组合优化', {'throughput': 5.0, 'energy_factor': 0.55, 'quality': 0.970}),
    ])

    def __init__(self, params_b, base_tokens_per_sec=80, gpu_tdp_w=400):
        self.params_b = params_b
        self.base_tokens_per_sec = base_tokens_per_sec
        self.gpu_tdp_w = gpu_tdp_w

    def tokens_per_joule(self, strategy):
        '''计算每焦耳生成 token 数'''
        cfg = self.STRATEGIES[strategy]
        tokens_per_sec = self.base_tokens_per_sec * cfg['throughput']
        power_w = self.gpu_tdp_w * cfg['energy_factor']
        return tokens_per_sec / power_w

    def analyze(self):
        results = OrderedDict()
        for name, cfg in self.STRATEGIES.items():
            tpj = self.tokens_per_joule(name)
            results[name] = OrderedDict([
                ('strategy', name),
                ('throughput', cfg['throughput']),
                ('tokens_per_sec', round(self.base_tokens_per_sec * cfg['throughput'], 1)),
                ('power_w', round(self.gpu_tdp_w * cfg['energy_factor'], 0)),
                ('tokens_per_joule', round(tpj, 3)),
                ('quality', cfg['quality']),
            ])
        return results

    def energy_for_tokens(self, strategy, n_tokens=1_000_000):
        '''生成指定数量 token 所需能耗（焦耳）'''
        return n_tokens / self.tokens_per_joule(strategy)

# 分析 7B 模型推理能效
analyzer = InferenceEfficiencyAnalyzer(params_b=7, base_tokens_per_sec=80, gpu_tdp_w=400)

print(f'\n--- 7B 模型推理能效对比 ---')
print(f"{'策略':<14}{'吞吐倍率':>10}{'tok/s':>9}{'功率(W)':>9}{'tok/J':>9}{'质量':>7}")
for name, info in analyzer.analyze().items():
    print(f"{name:<14}{info['throughput']:>9.2f}x{info['tokens_per_sec']:>9.1f}"
          f"{info['power_w']:>9.0f}{info['tokens_per_joule']:>9.3f}{info['quality']:>7.1%}")

# 生成 100 万 token 的能耗对比
print('\n--- 生成 100 万 token 所需能耗 ---')
print(f"{'策略':<14}{'能耗(kJ)':>12}{'相对基线':>12}")
baseline_energy = analyzer.energy_for_tokens('FP16 基线')
for name in analyzer.STRATEGIES:
    energy = analyzer.energy_for_tokens(name)
    ratio = energy / baseline_energy
    print(f"{name:<14}{energy/1000:>12.2f}{ratio:>11.2%}")

# 大规模服务场景：日均 10 亿次推理请求
print('\n--- 大规模服务场景（日均 10 亿次推理，每次 200 token）---')
daily_tokens = 1_000_000_000 * 200
print(f'  日均生成 token 数: {daily_tokens:.2e}')
print(f"{'策略':<14}{'日能耗(MWh)':>14}{'年CO2(吨)':>12}")
for name in ['FP16 基线', 'INT8 量化', 'INT4 量化', '组合优化']:
    energy_j = analyzer.energy_for_tokens(name, daily_tokens)
    energy_mwh = energy_j / 3.6e9
    co2_tons = energy_mwh * 1000 * 580 / 1000  # 中国平均碳强度
    print(f"{name:<14}{energy_mwh:>14.2f}{co2_tons:>12.1f}")

print(f'\nKey: 推理能效优化的杠杆远大于训练——规模化服务中组合优化可使每焦耳 token 产出提升 5-10 倍，年减排 CO2 可达数千吨；tokens/Joule 是衡量推理绿色度的核心指标')

## 4. 硬件选择与数据中心

硬件选型与数据中心运营直接决定单位算力的能耗与碳排放。新一代 AI 加速器在性能功耗比上持续进步，数据中心 PUE 也是绿色 AI 的关键变量。

**主流 AI 加速器对比：**

- **NVIDIA A100**：80GB HBM2e，TDP 400W，FP16 算力约 312 TFLOPS
- **NVIDIA H100**：80GB HBM3，TDP 700W，FP16 算力约 989 TFLOPS，支持 FP8
- **NVIDIA L40**：48GB GDDR6，TDP 300W，推理优化
- **AMD MI300X**：192GB HBM3，TDP 750W，大显存适合长上下文
- **Google TPU v5e**：推理专用，能效比突出

**数据中心关键指标：**

- **PUE（Power Usage Effectiveness）**：总能耗 / IT 能耗，越接近 1 越好
- **冷却方式**：风冷（PUE 1.4-1.6）、液冷（PUE 1.1-1.2）、浸没式（PUE < 1.1）
- **可再生能源占比**：水电、风光、核电可大幅降低碳强度
- **地理位置**：高纬度地区自然冷却，低碳电网地区优先
- **余热回收**：将数据中心废热用于区域供暖

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 硬件能效对比 HardwareComparator ===')

class HardwareComparator:
    '''硬件能效对比器：评估各加速器的性能功耗比与碳效率'''

    # 硬件规格：FP16 算力(TFLOPS)、TDP(W)、显存(GB)、带宽(GB/s)
    HARDWARE = OrderedDict([
        ('A100 80G',  {'fp16_tflops': 312, 'tdp_w': 400, 'mem_gb': 80,  'bandwidth_gbs': 2039, 'supports_fp8': False}),
        ('H100 80G',  {'fp16_tflops': 989, 'tdp_w': 700, 'mem_gb': 80,  'bandwidth_gbs': 3350, 'supports_fp8': True}),
        ('L40 48G',   {'fp16_tflops': 181, 'tdp_w': 300, 'mem_gb': 48,  'bandwidth_gbs': 864,  'supports_fp8': True}),
        ('MI300X',    {'fp16_tflops': 1307,'tdp_w': 750, 'mem_gb': 192, 'bandwidth_gbs': 5300, 'supports_fp8': True}),
        ('TPU v5e',   {'fp16_tflops': 197, 'tdp_w': 175, 'mem_gb': 16,  'bandwidth_gbs': 819,  'supports_fp8': False}),
    ])

    def __init__(self, pue=1.2, carbon_intensity=580):
        self.pue = pue
        self.carbon_intensity = carbon_intensity  # gCO2/kWh

    def perf_per_watt(self, name):
        '''每瓦特 FP16 算力（TFLOPS/W）'''
        hw = self.HARDWARE[name]
        return hw['fp16_tflops'] / hw['tdp_w']

    def energy_per_pflop(self, name):
        '''每 PFLOP(10^15) 计算所需能耗(kWh)'''
        hw = self.HARDWARE[name]
        effective_power = hw['tdp_w'] * self.pue
        pflops_per_sec = hw['fp16_tflops'] / 1000.0
        seconds_per_pflop = 1.0 / pflops_per_sec
        return effective_power * seconds_per_pflop / 3600.0

    def carbon_per_pflop(self, name):
        '''每 PFLOP 计算的碳排放(gCO2)'''
        return self.energy_per_pflop(name) * self.carbon_intensity

    def analyze(self):
        results = OrderedDict()
        for name in self.HARDWARE:
            hw = self.HARDWARE[name]
            results[name] = OrderedDict([
                ('name', name),
                ('fp16_tflops', hw['fp16_tflops']),
                ('tdp_w', hw['tdp_w']),
                ('mem_gb', hw['mem_gb']),
                ('perf_per_watt', round(self.perf_per_watt(name), 3)),
                ('energy_per_pflop_wh', round(self.energy_per_pflop(name) * 1000, 4)),
                ('carbon_per_pflop_g', round(self.carbon_per_pflop(name), 3)),
                ('supports_fp8', hw['supports_fp8']),
            ])
        return results

# 默认数据中心配置
comparator = HardwareComparator(pue=1.2, carbon_intensity=580)

print('\n--- AI 加速器能效对比（PUE=1.2, 碳强度=580）---')
print(f"{'硬件':<12}{'FP16(TF)':>10}{'TDP(W)':>8}{'显存(GB)':>10}{'TF/W':>7}{'Wh/PF':>9}{'gCO2/PF':>9}")
for name, info in comparator.analyze().items():
    print(f"{name:<12}{info['fp16_tflops']:>10}{info['tdp_w']:>8}{info['mem_gb']:>10}"
          f"{info['perf_per_watt']:>7.3f}{info['energy_per_pflop_wh']:>9.3f}{info['carbon_per_pflop_g']:>9.3f}")

# 不同 PUE 对总能耗的影响
print('\n--- 数据中心 PUE 对 H100 集群能耗影响 ---')
print(f"{'PUE':>6}{'有效TDP(W)':>12}{'Wh/PF':>10}{'gCO2/PF':>10}{'相对1.5':>10}")
base_carbon = None
for pue in [1.5, 1.3, 1.2, 1.1, 1.05]:
    cmp = HardwareComparator(pue=pue, carbon_intensity=580)
    energy = cmp.energy_per_pflop('H100 80G') * 1000
    carbon = cmp.carbon_per_pflop('H100 80G')
    if base_carbon is None:
        base_carbon = carbon
    ratio = carbon / base_carbon
    print(f"{pue:>6.2f}{700*pue:>12.0f}{energy:>10.3f}{carbon:>10.3f}{ratio:>9.1%}")

# 不同地区碳强度对 H100 集群碳效率
print('\n--- H100 集群在不同地区的碳效率 ---')
print(f"{'地区':<14}{'碳强度':>10}{'gCO2/PF':>10}{'相对澳洲':>10}")
regions = [('挪威', 30), ('法国', 60), ('美国西部', 250), ('中国平均', 610), ('澳大利亚', 700)]
ref = None
for region, ci in regions:
    cmp = HardwareComparator(pue=1.2, carbon_intensity=ci)
    carbon = cmp.carbon_per_pflop('H100 80G')
    if ref is None:
        ref = carbon
    ratio = carbon / ref
    print(f"{region:<14}{ci:>10}{carbon:>10.3f}{ratio:>9.1%}")

print(f'\nKey: H100 与 MI300X 在绝对性能上领先，但 TPU v5e 在性能功耗比上最优（适合推理）；液冷可将 PUE 从 1.5 降至 1.1，等效节能 27%；选址低碳电网地区可进一步将碳效率提升 10 倍以上')

## 5. 可持续 AI 实践

绿色 AI 不只是技术问题，更是工程实践与组织策略的综合。从训练调度到模型复用，从架构设计到评估标准，都需要纳入可持续性考量。

**核心实践：**

- **碳感知调度**：将训练任务调度到低碳地区与可再生能源充沛时段
- **模型复用**：优先 fine-tune 已有模型而非从头训练，避免重复能耗
- **高效架构**：Mamba、RWKV 等线性复杂度架构降低长序列能耗
- **模型共享**：开源生态减少重复造轮子
- **能效评估**：将能耗与碳排放纳入模型评估指标
- **生命周期管理**：从训练、部署到下线的全周期碳核算

**碳感知调度策略：**

- **空间维度**：选择低碳电网地区的数据中心
- **时间维度**：在风光发电高峰期（午间/夜间）执行高负载训练
- **弹性调度**：可中断训练任务跟随可再生能源波动
- **混合部署**：跨区域调度，平衡延迟与碳强度

In [ ]:
import torch
import torch.nn as nn
import math
from collections import OrderedDict

torch.manual_seed(42)

print('=== 可持续 AI 规划 SustainableAIPlanner ===')

class SustainableAIPlanner:
    '''可持续 AI 规划器：通过碳感知调度最小化训练碳排放'''

    # 各地区 24 小时碳强度曲线（gCO2/kWh），简化为 6 个时段
    REGION_PROFILES = OrderedDict([
        ('挪威', [28, 28, 30, 32, 30, 28]),       # 全天低碳（水电）
        ('中国华东', [620, 600, 580, 560, 580, 610]), # 午间略低（光伏）
        ('美国西部', [320, 280, 220, 200, 240, 300]), # 午间最低（太阳能）
        ('澳大利亚', [720, 680, 620, 600, 660, 710]), # 全天高碳
    ])

    def __init__(self, total_energy_kwh, hours_per_slot=4):
        self.total_energy_kwh = total_energy_kwh
        self.hours_per_slot = hours_per_slot  # 每个时段时长
        self.slots_per_day = 24 // hours_per_slot

    def slot_energy(self):
        '''每个时段所需能耗（均匀分配）'''
        return self.total_energy_kwh / self.slots_per_day

    def plan_static(self, region):
        '''静态调度：固定地区全天训练'''
        intensities = self.REGION_PROFILES[region]
        total_carbon = sum(self.slot_energy() * i for i in intensities) / 1000.0  # kg
        return OrderedDict([
            ('mode', '静态调度'),
            ('region', region),
            ('energy_kwh', round(self.total_energy_kwh, 1)),
            ('carbon_kg', round(total_carbon, 1)),
            ('carbon_tons', round(total_carbon / 1000, 3)),
        ])

    def plan_carbon_aware(self):
        '''碳感知调度：每个时段选择全球碳强度最低的地区'''
        total_carbon = 0.0
        schedule = []
        for slot_idx in range(self.slots_per_day):
            # 找到该时段碳强度最低的地区
            best_region = None
            best_intensity = float('inf')
            for region, intensities in self.REGION_PROFILES.items():
                if intensities[slot_idx] < best_intensity:
                    best_intensity = intensities[slot_idx]
                    best_region = region
            slot_energy = self.slot_energy()
            slot_carbon = slot_energy * best_intensity / 1000.0
            total_carbon += slot_carbon
            schedule.append({
                'slot': slot_idx,
                'region': best_region,
                'intensity': best_intensity,
                'energy_kwh': round(slot_energy, 1),
                'carbon_kg': round(slot_carbon, 1),
            })
        return OrderedDict([
            ('mode', '碳感知调度'),
            ('energy_kwh', round(self.total_energy_kwh, 1)),
            ('carbon_kg', round(total_carbon, 1)),
            ('carbon_tons', round(total_carbon / 1000, 3)),
            ('schedule', schedule),
        ])

    def plan_renewable_following(self, region='美国西部'):
        '''跟随可再生能源调度：仅在低碳时段训练'''
        intensities = self.REGION_PROFILES[region]
        avg_intensity = sum(intensities) / len(intensities)
        # 仅在低于平均碳强度的时段训练
        green_slots = [(i, v) for i, v in enumerate(intensities) if v < avg_intensity]
        if not green_slots:
            green_slots = [(0, intensities[0])]
        slot_energy = self.total_energy_kwh / len(green_slots)
        total_carbon = sum(slot_energy * v for _, v in green_slots) / 1000.0
        return OrderedDict([
            ('mode', '可再生能源跟随'),
            ('region', region),
            ('green_slots', len(green_slots)),
            ('energy_kwh', round(self.total_energy_kwh, 1)),
            ('carbon_kg', round(total_carbon, 1)),
            ('carbon_tons', round(total_carbon / 1000, 3)),
        ])

# 模拟 70B 模型训练：总能耗 500 MWh
planner = SustainableAIPlanner(total_energy_kwh=500_000, hours_per_slot=4)

print('\n--- 静态调度：固定地区全天训练 ---')
print(f"{'地区':<14}{'能耗(MWh)':>12}{'CO2(吨)':>10}")
static_results = {}
for region in planner.REGION_PROFILES:
    plan = planner.plan_static(region)
    static_results[region] = plan['carbon_tons']
    print(f"{region:<14}{plan['energy_kwh']/1000:>12.1f}{plan['carbon_tons']:>10.2f}")

print('\n--- 碳感知调度：每时段选全球最低碳地区 ---')
aware_plan = planner.plan_carbon_aware()
print(f"  总能耗: {aware_plan['energy_kwh']/1000:.1f} MWh")
print(f"  总碳排放: {aware_plan['carbon_tons']:.2f} 吨")
print(f"  相对澳大利亚减排: {(1 - aware_plan['carbon_tons']/static_results['澳大利亚']):.1%}")
print(f"\n  {'时段':>6}{'调度地区':<14}{'碳强度':>10}{'能耗(MWh)':>12}{'CO2(吨)':>10}")
for s in aware_plan['schedule']:
    print(f"  {s['slot']:>6}{s['region']:<14}{s['intensity']:>10}{s['energy_kwh']/1000:>12.1f}{s['carbon_kg']/1000:>10.2f}")

print('\n--- 可再生能源跟随调度 ---')
for region in ['美国西部', '中国华东']:
    plan = planner.plan_renewable_following(region)
    print(f"  {region}: 绿色时段 {plan['green_slots']} 个, CO2 {plan['carbon_tons']:.2f} 吨")

# 综合对比
print('\n--- 调度策略综合对比 ---')
print(f"{'策略':<24}{'CO2(吨)':>10}{'相对最差':>10}")
worst = max(static_results.values())
strategies = [
    ('静态-澳大利亚', static_results['澳大利亚']),
    ('静态-中国华东', static_results['中国华东']),
    ('静态-美国西部', static_results['美国西部']),
    ('静态-挪威', static_results['挪威']),
    ('碳感知全球调度', aware_plan['carbon_tons']),
]
for name, carbon in strategies:
    print(f"{name:<24}{carbon:>10.2f}{carbon/worst:>9.1%}")

print(f'\nKey: 碳感知调度是绿色 AI 最具杠杆的实践——通过跨地区、跨时段调度，可在不增加硬件成本的前提下将训练碳排放降低 80% 以上；叠加模型复用与高效架构，可持续 AI 可在不牺牲模型质量的同时实现数量级的碳减排')

## 📝 课后思考题

1. 假设你要训练一个 13B 模型，预算为 100 MWh 电力。你会如何选择训练地区、精度策略与硬件组合以最小化碳排放？请给出量化估算。
2. 推理能效优化中，INT4 量化与动态批处理分别从哪些维度提升 tokens/Joule？二者是否存在冲突？如何协同？
3. 碳感知调度要求训练任务可中断与跨地区迁移。这对训练框架（如 checkpoint 恢复、数据本地化）提出了哪些技术挑战？
4. 从全生命周期看，训练能耗与推理能耗哪个更大？在什么条件下推理总能耗会超过训练？这对绿色 AI 策略有何启示？